# 5. Free Energy: WHAM & MBAR

`genepie` exposes GENESIS's **WHAM** (Weighted Histogram Analysis Method) and
**MBAR** (Multistate Bennett Acceptance Ratio) estimators. Both reconstruct a
free-energy profile — a *potential of mean force* (PMF) — from **umbrella
sampling**: a set of simulations in which a harmonic restraint
$k\,(x - r_i)^2$ pins the reaction coordinate $x$ near a different center
$r_i$ in each window.

This chapter has two parts:

1. A **self-contained example** that generates synthetic umbrella data from a
   *known* PMF and shows that WHAM and MBAR recover it. This runs anywhere —
   no external data required.
2. The same two estimators applied to the **real GENESIS umbrella-sampling
   datasets** shipped in the source tree.

```{admonition} Inspired by MDToolbox.jl
:class: seealso
The worked example below follows the WHAM tutorial in
[MDToolbox.jl](https://www.bio.ics.saitama-u.ac.jp/MDToolbox.jl/dev/), a Julia
toolbox for MD trajectory analysis by the same group.
```


## A verifiable example: recover a known PMF

We pick a ground-truth profile — a single harmonic well
$U_0(x) = c\,(x - x_0)^2$ — and pretend we ran umbrella sampling on it. For a
harmonic $U_0$ each biased ensemble

$$p_i(x) \;\propto\; \exp\!\big(-\beta\,[\,U_0(x) + k\,(x - r_i)^2\,]\big)$$

is itself an exact Gaussian, so we can draw samples directly instead of running
a real simulation. Each window's samples are written to a GENESIS `cvfile`
(two columns: frame index and coordinate value).


In [ ]:
import numpy as np, tempfile, os
from genepie import genesis_exe

rng  = np.random.default_rng(0)
kB   = 0.0019872041          # Boltzmann constant (kcal/mol/K)
T    = 300.0
beta = 1.0 / (kB * T)

# --- Ground-truth free-energy profile we will try to recover ---
c, x0 = 0.30, 7.0            # U0(x) = c (x - x0)^2   [kcal/mol]
def true_pmf(x):
    return c * (x - x0) ** 2

# --- Umbrella windows: harmonic restraint  k (x - r_i)^2 ---
k          = 2.0                          # force constant (kcal/mol/A^2)
centers    = np.arange(3.0, 11.01, 0.5)   # 17 windows spanning the well
n_per_win  = 4000

tmp        = tempfile.mkdtemp()
cv_pattern = os.path.join(tmp, "{}.dat")  # "{}" expands to the window index
samples    = []
for i, r in enumerate(centers, start=1):
    A   = c + k
    mu  = (c * x0 + k * r) / A            # mean of the biased Gaussian
    sig = np.sqrt(1.0 / (2 * beta * A))   # its standard deviation
    xk  = rng.normal(mu, sig, size=n_per_win)
    samples.append(xk)
    np.savetxt(cv_pattern.format(i),
               np.column_stack([np.arange(1, n_per_win + 1), xk]),
               fmt=["%d", "%.6f"])

print(f"wrote {len(centers)} umbrella windows to {tmp}")

### Step 1 — the raw biased histograms

Each window only samples a narrow slice of the coordinate. Individually they say
nothing about the global free energy; the trick is that neighbouring windows
**overlap**.


In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"

from plotly.colors import sample_colorscale

shades = sample_colorscale("Turbo", np.linspace(0.05, 0.95, len(centers)))
fig = go.Figure()
for color, xk in zip(shades, samples):
    h, edges = np.histogram(xk, bins=40, range=(2, 12), density=True)
    ctr = 0.5 * (edges[:-1] + edges[1:])
    fig.add_trace(go.Scatter(x=ctr, y=h, mode="lines", showlegend=False,
                             line=dict(color=color, width=2)))
fig.update_layout(
    title=dict(text="<b>Biased distributions of the 17 umbrella windows</b>", font=dict(size=17)),
    xaxis_title="reaction coordinate x", yaxis_title="P<sub>i</sub>(x)",
    height=380, template="plotly_white",
    font=dict(family="Inter, Helvetica, Arial, sans-serif", size=13, color="#333"),
    margin=dict(l=60, r=30, t=60, b=50),
)
fig

### Step 2 — WHAM stitches them into one PMF

`wham_analysis` unbiases and combines the histograms. We pass the same force
constant `constant` and centers `reference` that defined the restraints. The
result is one row per grid bin: column 0 is the bin center, column 1 the free
energy in kcal/mol.


In [ ]:
pmf = genesis_exe.wham_analysis(
    cvfile=cv_pattern, dimension=1, nblocks=1, temperature=T, tolerance=1e-8,
    rest_function=(1,), grids=((2.0, 12.0, 101),),
    constant=(tuple([k] * len(centers)),),
    reference=(tuple(centers),), is_periodic=(False,),
)
x, f  = pmf[:, 0], pmf[:, 1]
f      = f - np.nanmin(f)                 # shift minimum to zero
truth  = true_pmf(x); truth -= truth.min()

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=truth, mode="lines", name="true PMF",
                         line=dict(width=8, color="#D9D9D9")))
fig.add_trace(go.Scatter(x=x, y=f, mode="markers", name="WHAM estimate",
                         marker=dict(size=7, color="#C44E52",
                                     line=dict(width=1, color="white"))))
fig.update_layout(
    title=dict(text="<b>WHAM recovers the ground-truth free-energy profile</b>", font=dict(size=17)),
    xaxis_title="reaction coordinate x", yaxis_title="free energy (kcal/mol)",
    height=400, template="plotly_white",
    font=dict(family="Inter, Helvetica, Arial, sans-serif", size=13, color="#333"),
    legend=dict(orientation="h", y=1.02, yanchor="bottom"),
    margin=dict(l=60, r=30, t=60, b=50),
)
fig

In [ ]:
# Quantitative check: fit the recovered PMF to a parabola near the well
sel   = (x > 4) & (x < 10)
coeff = np.polyfit(x[sel], f[sel], 2)
print(f"recovered curvature = {coeff[0]:.3f} kcal/mol/A^2   (true c = {c})")
print(f"recovered minimum   = {-coeff[1]/(2*coeff[0]):.2f}            (true x0 = {x0})")

### Step 3 — MBAR gives the same answer without binning

MBAR works directly on the samples (no histogram). For umbrella sampling it
returns the **relative free energy of each window**, in reduced units of
$k_\mathrm{B}T$. For our harmonic ground truth these follow the analytical curve
$f_k = \beta\,\frac{c\,k}{c+k}\,(r_i - x_0)^2$.


In [ ]:
fene = genesis_exe.mbar_analysis(
    cvfile=cv_pattern, nreplica=len(centers), input_type="US", dimension=1,
    temperature=T, target_temperature=T, tolerance=1e-8, rest_function=(1,),
    grids=((2.0, 12.0, 101),),
    constant=(tuple([k] * len(centers)),),
    reference=(tuple(centers),), is_periodic=(False,), box_size=(0.0,),
)
fene = np.asarray(fene).ravel()
fene = fene - fene.min()

A   = c + k
ana = beta * (c * k / A) * (centers - x0) ** 2      # reduced units (k_B T)
ana = ana - ana.min()

fig = go.Figure()
fig.add_trace(go.Scatter(x=centers, y=ana, mode="lines", name="analytical",
                         line=dict(width=8, color="#D9D9D9")))
fig.add_trace(go.Scatter(x=centers, y=fene, mode="markers", name="MBAR",
                         marker=dict(size=9, color="#4C72B0",
                                     line=dict(width=1, color="white"))))
fig.update_layout(
    title=dict(text="<b>MBAR window free energies vs. the analytical result</b>", font=dict(size=17)),
    xaxis_title="umbrella center r",
    yaxis_title="reduced free energy f<sub>k</sub>  (units of k<sub>B</sub>T)",
    height=400, template="plotly_white",
    font=dict(family="Inter, Helvetica, Arial, sans-serif", size=13, color="#333"),
    legend=dict(orientation="h", y=1.02, yanchor="bottom"),
    margin=dict(l=60, r=30, t=60, b=50),
)
fig

## The same estimators on real GENESIS data

Now the identical API on the umbrella-sampling reference datasets that ship in
the source tree (`tests/regression_test/`).

```{admonition} Data
:class: note
These datasets are present in a source checkout / CI build. If you installed
only the wheel, the cells below detect the missing data and skip gracefully.
```


In [ ]:
import genepie, pathlib
REPO_ROOT = pathlib.Path(genepie.__file__).resolve().parents[2]
REG       = REPO_ROOT / "tests" / "regression_test"
HAVE_REG  = REG.is_dir()
print("regression data available:", HAVE_REG)

### `cvfile` is required and validated up front

`cvfile` is a filename *pattern* where `{}` expands to the window/replica index
(e.g. `"run{}.dis"`). It is mandatory, and its existence is checked **before**
GENESIS runs, so a typo raises a catchable `GenesisValidationError` instead of
aborting the interpreter.


In [ ]:
from genepie.exceptions import GenesisValidationError

try:
    genesis_exe.wham_analysis(
        cvfile=None, dimension=1, nblocks=1, temperature=300.0,
        tolerance=1e-8, rest_function=(1,), grids=((0.0, 15.0, 301),),
        constant=((1.2,) * 14,),
        reference=((1.8, 2.72, 3.64, 4.56, 5.48, 6.4, 7.32, 8.24, 9.16,
                    10.08, 11.0, 11.92, 12.84, 13.76),),
        is_periodic=(False,),
    )
except GenesisValidationError as e:
    print("Caught GenesisValidationError:", e)

### WHAM: 1-D PMF of trialanine

Fourteen umbrella windows along the end-to-end distance of trialanine, sampled
with real GENESIS simulations (40&#8239;000 frames per window). The API call is
identical to the synthetic example above — only the `cvfile` pattern and the
window definitions change.


In [ ]:
CONSTANT  = (1.2,) * 14
REFERENCE = (1.80, 2.72, 3.64, 4.56, 5.48, 6.40, 7.32,
             8.24, 9.16, 10.08, 11.00, 11.92, 12.84, 13.76)

pmf_real = None
if HAVE_REG:
    cvfile   = str(REG / "test_analysis" / "trajectories" / "triala_cv" / "{}.dis")
    pmf_real = genesis_exe.wham_analysis(
        cvfile=cvfile, dimension=1, nblocks=1, temperature=300.0,
        tolerance=1e-8, rest_function=(1,), grids=((0.0, 15.0, 301),),
        constant=(CONSTANT,), reference=(REFERENCE,), is_periodic=(False,),
    )
    print("PMF shape (bins, columns):", pmf_real.shape)
    print("free-energy range:", round(float(pmf_real[:, 1].min()), 3),
          "to", round(float(pmf_real[:, 1].max()), 3), "kcal/mol")
else:
    print("regression data not found; skipping WHAM computation")

In [ ]:
fig = go.Figure()
if pmf_real is not None:
    m = np.isfinite(pmf_real[:, 1])
    y = pmf_real[m, 1] - np.nanmin(pmf_real[m, 1])
    fig.add_trace(go.Scatter(x=pmf_real[m, 0], y=y, mode="lines",
                             line=dict(color="#C44E52", width=3),
                             fill="tozeroy", fillcolor="rgba(196,78,82,0.10)"))
fig.update_layout(
    title=dict(text="<b>WHAM PMF of trialanine (real GENESIS data)</b>", font=dict(size=17)),
    xaxis_title="end-to-end distance (&#8491;)", yaxis_title="free energy (kcal/mol)",
    height=400, template="plotly_white",
    font=dict(family="Inter, Helvetica, Arial, sans-serif", size=13, color="#333"),
    margin=dict(l=60, r=30, t=60, b=50),
)
fig

### MBAR on a periodic dihedral

GENESIS also ships a periodic torsion dataset (`umbrella_1d/`, 61 windows spaced
3&deg; apart). MBAR handles it through the same interface — pass `is_periodic=(True,)`
and `box_size=(360.0,)`.

```{admonition} Run many solves in one session with the *_isolated wrappers
:class: tip
GENESIS keeps global Fortran state (solver work arrays, accumulated counters)
across calls, so running four or more WHAM/MBAR solves back-to-back in a
*single* Python session used to eventually crash the kernel. `wham_analysis_isolated`
and `mbar_analysis_isolated` run each solve in a throwaway subprocess, giving
every call clean state. They accept the same arguments as `wham_analysis` /
`mbar_analysis` (plus an optional `timeout`) and return the same arrays, so any
number of estimates can be computed in one kernel session. The cell below is the
fourth solve of this chapter and executes safely at build time thanks to that
isolation.
```


In [ ]:
DIHEDRAL_CENTERS = np.array([3.0 * i for i in range(61)])

fene_real = None
if HAVE_REG:
    mbar_cv = str(REG / "test_analysis" / "trajectories" / "umbrella_1d" / "{}.dat")
    # *_isolated runs the solve in a fresh subprocess, so this fourth solve of
    # the chapter cannot exhaust the accumulated Fortran state.
    fene_real = genesis_exe.mbar_analysis_isolated(
        cvfile=mbar_cv, nreplica=61, input_type="US", dimension=1,
        temperature=300.0, target_temperature=300.0, tolerance=1e-8,
        rest_function=(1,), grids=((-1.0, 181.0, 81),),
        constant=((0.06092,) * 61,),
        reference=(tuple(DIHEDRAL_CENTERS),),
        is_periodic=(True,), box_size=(360.0,),
    )
    fene_real = np.asarray(fene_real).ravel()
    fene_real = fene_real - fene_real.min()
    print("MBAR free energies (reduced units), shape:", fene_real.shape)
    print("range:", round(float(fene_real.min()), 3), "to",
          round(float(fene_real.max()), 3), "k_BT")
else:
    print("regression data not found; skipping MBAR computation")

In [ ]:
fig = go.Figure()
if fene_real is not None:
    fig.add_trace(go.Scatter(x=DIHEDRAL_CENTERS, y=fene_real, mode="lines+markers",
                             line=dict(color="#4C72B0", width=2),
                             marker=dict(size=8, color="#4C72B0",
                                         line=dict(width=1, color="white"))))
fig.update_layout(
    title=dict(text="<b>MBAR window free energies of the periodic dihedral (real GENESIS data)</b>",
               font=dict(size=16)),
    xaxis_title="umbrella center (degrees)",
    yaxis_title="reduced free energy f<sub>k</sub>  (units of k<sub>B</sub>T)",
    height=400, template="plotly_white",
    font=dict(family="Inter, Helvetica, Arial, sans-serif", size=13, color="#333"),
    margin=dict(l=60, r=30, t=60, b=50),
)
fig

## From weights to a PMF: `pmf_analysis`

WHAM returns the PMF directly, and MBAR returns per-state (per-window) free
energies together with optional per-sample weights. When you already have the
reaction-coordinate samples and their weights — for example the MBAR weights of
a T-REMD run — `genepie` can turn them straight into a potential of mean force
with **`pmf_analysis`**, without any manual histogram or `-k_B T \log P` step:

```python
from genepie import genesis_exe

# 1-D PMF from samples + optional per-sample weights
result = genesis_exe.pmf_analysis(
    cv=cv_values,              # shape (n_sample,)
    weight=mbar_weights,       # optional; omit for an unweighted estimate
    temperature=300.0,
    grids=((0.0, 15.0, 101),), # (min, max, num_grids) per dimension
    band_width=(0.3,),         # Gaussian-kernel sigma per dimension
    is_periodic=(False,),
)
result.cv          # bin centers
result.pmf         # free energy (standard histogram estimator)
result.pmf_gaussian  # free energy (Gaussian-kernel estimator)
```

For a 2-D reaction coordinate, pass `cv` of shape `(n_sample, 2)` and one entry
per dimension in `grids`/`band_width`; the result then carries `cv1`, `cv2`, and
a `pmf` matrix. It also accepts CLI-style `cvfile`/`weightfile` filename
patterns instead of in-memory arrays. See
[5b. MBAR Weighted Resampling](05b_mbar_resampling.ipynb) for a worked
Ramachandran example that builds a reweighted surface this way.
